# Part 2 - Model Training with PyTorch

In this tutorial, we will create and train a neural network model using **PyTorch 2.x** - a modern deep learning framework.

## What's New in PyTorch Version

This notebook has been migrated from TensorFlow/TFLearn to PyTorch for several reasons:
- **Better Performance**: PyTorch 2.x offers improved performance with torch.compile
- **Smaller Size**: PyTorch CPU is ~200MB vs TensorFlow's ~600MB
- **Modern Architecture**: Uses EfficientNet-B0 + LSTM for temporal awareness
- **Active Development**: PyTorch has a larger active community

## Available Model Architectures

| Model | Description | Parameters | Temporal |
|-------|-------------|-----------|----------|
| `efficientnet_lstm` | **RECOMMENDED** - EfficientNet-B0 + LSTM | ~5M | Yes |
| `mobilenet_v3` | Ultra-fast for low-end hardware | ~2M | No |
| `resnet18_lstm` | Good balance of speed/accuracy | ~12M | Yes |
| `inception_v3` | Legacy (ported from TFLearn) | ~7M | No |
| `alexnet` | Legacy (ported from TFLearn) | ~60M | No |

## Loading Libraries

Here we load the required libraries for PyTorch-based training.

**PyTorch** is the deep learning framework that will handle our neural network operations.
- `torch`: Core PyTorch library
- `torch.nn`: Neural network modules
- `torch.optim`: Optimizers (AdamW, etc.)
- `DataLoader`: Efficient data loading with batching

In [ ]:
# Standard libraries
import numpy as np
import cv2
import os
import time
import pandas as pd
from pathlib import Path
from collections import deque
from random import shuffle

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Local imports - PyTorch models
from models_pytorch import (
    get_model, list_models, get_model_info, 
    save_model, load_model, count_parameters, get_device
)

# Check PyTorch version and device
print(f"PyTorch Version: {torch.__version__}")
device = get_device()
print(f"Using device: {device}")

## Available Models

Let's see what models are available for training:

In [ ]:
# List all available models
print("Available Models:")
print("=" * 60)
for name in list_models():
    info = get_model_info(name)
    rec = " [RECOMMENDED]" if info.get("recommended") else ""
    print(f"\n  {name}{rec}")
    print(f"    {info['description']}")
    print(f"    Params: {info['params']}, Inference: {info['inference_ms']}")

In [ ]:
# Analysis of the Input file
# We are going to analyze the files that we have created in Part 1
# Let us first select the first created file.

# Preprocessed image rgb color - no image filters
file_name = "preprocessed_training_data-1.npy"
# Alternative: file_name = "training_data-1.npy"

print(f"Loading data file: {file_name}")

In [4]:
# full file info
train_data = np.load(file_name,allow_pickle=True)

In [5]:
# This file has the following shape
train_data.shape
#(500, 2)

(500, 2)

In [6]:
#train_data

In [7]:
type(train_data )

numpy.ndarray

The the first three input frames are presented as:

In [8]:
train_data[0][1]

array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0])

In [9]:
train_data[1][1]

array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0])

In [10]:
train_data[2][1]

array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0])

There are 29 input componentes for each frame, we can plot the histogram for each component

In [11]:
train_hist = train_data[:]

In [12]:
train_hist.shape

(500, 2)

In [13]:
df = pd.DataFrame()
for i in range(len(train_hist)):
    row=list(train_hist[i][1])
    #print(row)
    temp = pd.DataFrame([row])
   # print(temp)
    df = pd.concat([df, temp])
  

In [14]:
df

,0,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


We can analize the train data

In [16]:
#ax = df.hist(bins=29, grid=False, figsize=(15,18), color='#86bf91', zorder=5, rwidth=5)

In [17]:
train = train_data[:-50]
test = train_data[-50:]

In [18]:
#test

In [19]:
test.shape
#(50, 2)

(50, 2)

In [20]:
train.shape
#(450, 2)

(450, 2)

We begin the train part

## Train Image part ( 4 Dimensional)

In [21]:
X_image = np.array([i[0] for i in train])

In [22]:
X_image.shape
#(450, 270, 480, 3)

(450, 270, 480, 3)

In [23]:
X_image[449][269][3]

array([0, 0, 0], dtype=uint8)

In [24]:
#(450, 270, 480, 3)  We choose only the imagen part of the train data, 
#There are 450 picturtes with resolutionn WIDTH = 480 and HEIGHT = 270 with 3 colors rgb

In [25]:
# We perform the reshape

numpy.reshape(a, newshape, order='C')

- a - Array to be reshaped.
- newshape  - The new shape should be compatible with the original shape.

- order- Read the elements of a using this index order, and place the elements into the reshaped array using this index order.

Gives a new shape to an array without changing its data.

In [26]:
WIDTH = 480
HEIGHT = 270

Using arr.reshape() will give a new shape to an array without changing the data. Just remember that when you use the reshape method, the array you want to produce needs to have the same number of elements as the original array.

If you start with an array with N elements, you’ll need to make sure that your new array also has a total of N elements

In [27]:
import numpy as np

In [28]:
X_image.size

174960000

In [29]:
X_image.ndim

4

In [30]:
X_image.shape

(450, 270, 480, 3)

You can use reshape() to reshape your array. 



![title](np_reshape.png)

With np.reshape, you can specify a few optional parameters:
np.reshape(a, newshape=(d, e), order='C')

a is the array to be reshaped.

newshape is the new shape you want. You can specify an integer or a tuple of integers. If you specify an integer, the result will be an array of that length. The shape should be compatible with the original shape.

order: C means to read/write the elements using C-like index order, F means to read/write the elements using Fortran-like index order, A means to read/write the elements in Fortran-like index order if a is Fortran contiguous in memory, C-like order otherwise. (This is an optional parameter and doesn’t need to be specified.)

If you want to learn more about C and Fortran order, you can read more about the internal organization of NumPy arrays here. Essentially, C and Fortran orders have to do with how indices correspond to the order the array is stored in memory. In Fortran, when moving through the elements of a two-dimensional array as it is stored in memory, the first index is the most rapidly varying index. As the first index moves to the next row as it changes, the matrix is stored one column at a time. This is why Fortran is thought of as a Column-major language. In C on the other hand, the last index changes the most rapidly. The matrix is stored by rows, making it a Row-major language. What you do for C or Fortran depends on whether it’s more important to preserve the indexing convention or not reorder the data.

In [31]:
X_image.size

174960000

We will reshape  270, 480 to  480, 270

(450, 270, 480, 3) -> (450, 480, 270, 3)

What does -1 mean in numpy reshape? A numpy matrix can be reshaped into a vector using reshape function with parameter -1. The criterion to satisfy for providing the new shape is that 'The new shape should be compatible with the original shape'

numpy allow us to give one of new shape parameter as -1 (eg: (-1,WIDTH,HEIGHT,3) . It simply means that it is an unknown dimension and we want numpy to figure it out. And numpy will figure this by looking at the 'length of the array and remaining dimensions' and making sure it satisfies the above mentioned criteria

In [32]:
# For preprocessed rgb
X=X_image.reshape(-1,WIDTH,HEIGHT,3)
X.shape
#(450, 480, 270, 3) For 3 colors

(450, 480, 270, 3)

In [33]:
X.size

174960000

In [34]:
# For processed unicolor 
#X=X_image.reshape(-1,WIDTH,HEIGHT,1)
#X.shape
#(450, 480, 270, 1) For one color

## Train Input part ( 1 Dimensional )

In [35]:
Y = [i[1] for i in train]
#Z = [i[2] for i in train]

In [38]:
Y[0]

array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0])

In [39]:
type(Y[0])

numpy.ndarray

In [36]:
len(Y)

450

In [105]:
type(Y)

list

We begin the test part

## Test Image part ( 4 Dimensional)

In [109]:
 test.shape

(50, 2)

In [91]:
test_image = np.array([i[0] for i in test])

In [100]:
type(test_image)

numpy.ndarray

In [99]:
test_image.ndim

4

In [93]:
test_image.shape
#(50, 270, 480, 3)

(50, 270, 480, 3)

numpy.reshape(a, newshape, order='C')

- a - Array to be reshaped.
- newshape  - The new shape should be compatible with the original shape.

- order- Read the elements of a using this index order, and place the elements into the reshaped array using this index order.

Gives a new shape to an array without changing its data.

In [95]:
#For Preprocessed
test_x=test_image.reshape(-1,WIDTH,HEIGHT,3)
test_x.shape
#(50, 480, 270, 3)

(50, 480, 270, 3)

In [96]:
#For Processed
#test_x=test_image.reshape(-1,WIDTH,HEIGHT,1)
#test_x.shape

## Test Input part

In [35]:
test_y = [i[1] for i in test]

In [ ]:
# =============================================================================
# Training Configuration
# =============================================================================

# Model configuration
MODEL_NAME = 'efficientnet_lstm'  # Change to any model from list_models()
NUM_ACTIONS = 29                   # 9 keyboard + 20 gamepad actions
TEMPORAL_FRAMES = 4                # Number of frames for temporal context (LSTM models)

# Training hyperparameters
FILE_I_END = 2                     # Number of data files to use (increase for full training)
EPOCHS = 1                         # Number of epochs per file
BATCH_SIZE = 16                    # Batch size for training
LEARNING_RATE = 1e-4               # Learning rate for AdamW optimizer
VAL_SPLIT = 0.1                    # Validation split ratio

# Paths
MODEL_DIR = Path('model')
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / f'{MODEL_NAME}.pth'

# Image dimensions
WIDTH = 480
HEIGHT = 270

# Previous model (set to load a checkpoint)
PREV_MODEL = ''
LOAD_MODEL = False

# Action labels (for reference)
ACTION_LABELS = [
    'W', 'S', 'A', 'D', 'WA', 'WD', 'SA', 'SD', 'NOKEY',     # Keyboard (0-8)
    'LT', 'RT', 'Lx', 'Ly', 'Rx', 'Ry',                      # Gamepad triggers/sticks (9-14)
    'UP', 'DOWN', 'LEFT', 'RIGHT', 'START', 'SELECT',        # Gamepad D-pad/buttons (15-20)
    'L3', 'R3', 'LB', 'RB', 'A', 'B', 'X', 'Y'               # Gamepad buttons (21-28)
]

print(f"Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Actions: {NUM_ACTIONS}")
print(f"  Temporal frames: {TEMPORAL_FRAMES}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")

In [37]:
len(nk )

9

In [38]:
#model = googlenet(WIDTH, HEIGHT, 3, LR, output=9, model_name=MODEL_NAME)

In [39]:
size=[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0]

In [40]:
len(size)

29

In [ ]:
# =============================================================================
# Create PyTorch Model
# =============================================================================

# Get model info
model_info = get_model_info(MODEL_NAME)
is_temporal = model_info.get("temporal", False)
seq_len = TEMPORAL_FRAMES if is_temporal else 1

print(f"Creating model: {MODEL_NAME}")
print(f"  Temporal: {is_temporal} (seq_len={seq_len})")

# Create model
model = get_model(
    MODEL_NAME,
    num_actions=NUM_ACTIONS,
    temporal_frames=seq_len,
    pretrained=True  # Use pretrained ImageNet weights for backbone
)

# Move to device
model = model.to(device)

# Count parameters
num_params = count_parameters(model)
print(f"  Parameters: {num_params:,}")

# Load previous model if specified
if LOAD_MODEL and PREV_MODEL and Path(PREV_MODEL).exists():
    model, metadata = load_model(PREV_MODEL, device=device)
    print(f"Loaded previous model from: {PREV_MODEL}")

In [ ]:
# =============================================================================
# PyTorch Dataset and DataLoader
# =============================================================================

class GameplayDataset(Dataset):
    """PyTorch Dataset for gameplay recordings."""
    
    def __init__(self, frames, actions, seq_len=1):
        """
        Args:
            frames: numpy array of frames (N, H, W, C)
            actions: numpy array of action labels (N, num_actions)
            seq_len: sequence length for temporal models
        """
        self.frames = frames
        self.actions = actions.astype(np.float32)
        self.seq_len = seq_len
    
    def __len__(self):
        return max(0, len(self.frames) - self.seq_len + 1)
    
    def __getitem__(self, idx):
        # Get sequence of frames
        if self.seq_len > 1:
            frames = self.frames[idx:idx + self.seq_len]
        else:
            frames = self.frames[idx:idx + 1]
        
        # Get action for last frame in sequence
        action = self.actions[idx + self.seq_len - 1]
        
        # Convert to tensor (NCHW format for PyTorch)
        frames = torch.tensor(frames, dtype=torch.float32)
        frames = frames.permute(0, 3, 1, 2)  # (seq, H, W, C) -> (seq, C, H, W)
        frames = frames / 255.0  # Normalize to [0, 1]
        
        # Remove sequence dim if seq_len == 1
        if self.seq_len == 1:
            frames = frames.squeeze(0)
        
        action = torch.tensor(action, dtype=torch.float32)
        
        return frames, action

# Quick test with current data
print(f"Creating dataset with seq_len={seq_len}")
dataset = GameplayDataset(X_image, np.array(Y), seq_len=seq_len)
print(f"Dataset size: {len(dataset)} samples")

In [43]:
from IPython.display import display_html
def restartkernel() :
    display_html("<script>Jupyter.notebook.kernel.restart()</script>",raw=True)

In [ ]:
restartkernel()

## Full code

In [ ]:
# =============================================================================
# Complete PyTorch Training Loop
# =============================================================================

def train_epoch(model, dataloader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    
    for batch_idx, (frames, actions) in enumerate(dataloader):
        frames = frames.to(device)
        actions = actions.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, actions)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for frames, actions in dataloader:
            frames = frames.to(device)
            actions = actions.to(device)
            
            outputs = model(frames)
            loss = criterion(outputs, actions)
            total_loss += loss.item()
            
            # Calculate accuracy (multi-label)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == actions).all(dim=1).sum().item()
            total += actions.size(0)
    
    avg_loss = total_loss / len(dataloader) if len(dataloader) > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0
    
    return avg_loss, accuracy

# Setup optimizer and loss function
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * FILE_I_END)
criterion = nn.BCEWithLogitsLoss()  # Multi-label classification

print("="*60)
print("Starting Training")
print("="*60)
print(f"  Model: {MODEL_NAME}")
print(f"  Device: {device}")
print(f"  Epochs per file: {EPOCHS}")
print(f"  Data files: {FILE_I_END}")
print("="*60)

best_val_loss = float('inf')

# Training loop
for epoch in range(EPOCHS):
    data_order = list(range(1, FILE_I_END + 1))
    shuffle(data_order)
    
    for count, i in enumerate(data_order):
        try:
            # Load data file
            file_name = f'preprocessed_training_data-{i}.npy'
            train_data = np.load(file_name, allow_pickle=True)
            
            # Split train/validation
            train = train_data[:-50]
            test = train_data[-50:]
            
            # Extract frames and actions
            train_frames = np.array([x[0] for x in train])
            train_actions = np.array([x[1] for x in train])
            test_frames = np.array([x[0] for x in test])
            test_actions = np.array([x[1] for x in test])
            
            # Create datasets and dataloaders
            train_dataset = GameplayDataset(train_frames, train_actions, seq_len=seq_len)
            val_dataset = GameplayDataset(test_frames, test_actions, seq_len=seq_len)
            
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
            
            # Train epoch
            train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
            val_loss, val_acc = validate(model, val_loader, criterion, device)
            
            # Step scheduler
            scheduler.step()
            
            print(f"File {i}/{FILE_I_END} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
            
            # Save best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                save_model(model, str(MODEL_PATH), optimizer=optimizer, epoch=epoch, loss=val_loss)
                print(f"  -> Saved best model to {MODEL_PATH}")
            
            # Periodic checkpoint
            if count % 10 == 0:
                checkpoint_path = MODEL_DIR / f'{MODEL_NAME}_checkpoint.pth'
                save_model(model, str(checkpoint_path), optimizer=optimizer, epoch=epoch)
                print(f"  -> Checkpoint saved")
                
        except Exception as e:
            print(f"Error processing file {i}: {e}")

print("="*60)
print("Training Complete!")
print(f"Best model saved to: {MODEL_PATH}")
print("="*60)

## Alternative Approaches with PyTorch

In this notebook we used EfficientNet-B0 with LSTM, which is the **recommended** architecture for game bot AI. However, you can easily switch to other models by changing the `MODEL_NAME` variable.

### Available PyTorch Models

| Model | Size | Parameters | Temporal | Speed | Use Case |
|-------|------|-----------|----------|-------|----------|
| **efficientnet_lstm** | ~20MB | ~5M | Yes | Medium | Best accuracy with temporal awareness |
| **efficientnet_simple** | ~15MB | ~4M | No | Fast | Single-frame predictions |
| **mobilenet_v3** | ~10MB | ~2M | No | Fastest | Low-end hardware / real-time |
| **resnet18_lstm** | ~50MB | ~12M | Yes | Medium | Good balance of speed/accuracy |
| **inception_v3** | ~30MB | ~7M | No | Slow | Legacy (ported from TFLearn) |
| **alexnet** | ~230MB | ~60M | No | Medium | Legacy (ported from TFLearn) |
| **sentnet** | ~280MB | ~70M | Yes | Slow | Legacy 3D CNN |
| **sentnet_2d** | ~280MB | ~70M | No | Slow | Legacy 2D variant |

### Recommended Models by Use Case

1. **Best Overall**: `efficientnet_lstm` - Uses pretrained EfficientNet-B0 with LSTM for temporal context
2. **Fastest Inference**: `mobilenet_v3` - Optimized for real-time performance
3. **Memory Constrained**: `mobilenet_v3` - Smallest model size
4. **Legacy Compatibility**: `inception_v3` - Same architecture as original TFLearn version

### Why We Migrated to PyTorch

- **TFLearn is deprecated** - Last release was 2019, no TensorFlow 2.x support
- **Smaller bundle size** - PyTorch CPU is ~200MB vs TensorFlow's ~600MB
- **Better error messages** - PyTorch provides clearer debugging information
- **Modern features** - torch.compile, mixed precision, etc.
- **Active community** - More tutorials, examples, and support available